In [1]:

from pathlib import Path
from scipy.io import loadmat
import numpy as np

ROOT = (
    Path.home()
    / "Desktop"
    / "MachineFailurePredictor"
    / "data"
    / "raw"
    / "paderborn"
)

def unwrap(value):
    """Unwrap nested MATLAB values safely."""
    while isinstance(value, np.ndarray) and value.size == 1:
        value = value.item()
    return value

def get_signals(file_path):
    mat = loadmat(str(file_path))
    record = mat[file_path.stem][0, 0]
    signals = {}

    channels = record["Y"].reshape(-1)

    for channel in channels:
        name = str(unwrap(channel["Name"])).strip()

        raw = unwrap(channel["Data"])
        values = np.asarray(raw, dtype=float).squeeze()

        if values.ndim == 1 and values.size > 0:
            signals[name] = values

    return signals

files = sorted(ROOT.rglob("*.mat"))
print("MAT files:", len(files))

if not files:
    raise FileNotFoundError(f"No MAT files found in {ROOT}")

sample = get_signals(files[0])

for name, values in sample.items():
    print(
        name,
        "| samples:", len(values),
        "| first values:", values[:3]
    )


MAT files: 800
force | samples: 16008 | first values: [927.26279233 919.93857359 949.54062435]
phase_current_1 | samples: 256823 | first values: [-0.87268646 -0.94569733 -1.03799408]
phase_current_2 | samples: 256823 | first values: [-1.14613281 -1.24531738 -1.36860931]
speed | samples: 16008 | first values: [899.47424449 899.47442559 899.47460714]
temp_2_bearing_module | samples: 5 | first values: [46.78039548 47.00927731 46.98791501]
torque | samples: 16008 | first values: [1.14071496 1.14883383 1.14558032]
vibration_1 | samples: 256823 | first values: [ 0.06408691 -0.12207031  0.11291504]


In [2]:

import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew

def calculate_features(signal, prefix):
    signal = np.asarray(signal, dtype=float)
    signal = signal[np.isfinite(signal)]

    if len(signal) == 0:
        return {}

    return {
        f"{prefix}_mean": float(np.mean(signal)),
        f"{prefix}_std": float(np.std(signal)),
        f"{prefix}_rms": float(np.sqrt(np.mean(signal ** 2))),
        f"{prefix}_peak_to_peak": float(np.ptp(signal)),
        f"{prefix}_skewness": float(skew(signal)),
        f"{prefix}_kurtosis": float(kurtosis(signal)),
    }

def get_bearing_id(path):
    return path.stem.rsplit("_", 1)[0].split("_")[-1]

rows = []

for i, path in enumerate(files, start=1):
    try:
        signals = get_signals(path)

        row = {
            "file": path.name,
            "bearing_id": get_bearing_id(path),
        }

        for signal_name in [
            "vibration_1",
            "phase_current_1",
            "phase_current_2",
        ]:
            if signal_name in signals:
                row.update(
                    calculate_features(
                        signals[signal_name],
                        signal_name
                    )
                )

        rows.append(row)

    except Exception as error:
        print("Skipped:", path.name, "|", repr(error))

    if i % 100 == 0:
        print(f"Processed {i}/{len(files)} files")

features_df = pd.DataFrame(rows)

print("\nFeature table shape:", features_df.shape)
display(features_df.head())


Processed 100/800 files
Processed 200/800 files
Processed 300/800 files
Processed 400/800 files
Processed 500/800 files
Processed 600/800 files
Processed 700/800 files
Processed 800/800 files

Feature table shape: (800, 20)


,file,bearing_id,vibration_1_mean,vibration_1_std,vibration_1_rms,vibration_1_peak_to_peak,vibration_1_skewness,vibration_1_kurtosis,phase_current_1_mean,phase_current_1_std,phase_current_1_rms,phase_current_1_peak_to_peak,phase_current_1_skewness,phase_current_1_kurtosis,phase_current_2_mean,phase_current_2_std,phase_current_2_rms,phase_current_2_peak_to_peak,phase_current_2_skewness,phase_current_2_kurtosis
0,N09_M07_F10_K001_1.mat,K001,-0.015454,0.367295,0.367620,9.570312,-0.016718,17.165065,0.023506,1.656556,1.656722,5.730664,0.006718,-1.493523,0.074766,1.666412,1.668088,5.519897,0.008774,-1.494215
1,N09_M07_F10_K001_10.mat,K001,-0.015243,0.362781,0.363101,8.596802,-0.005482,15.989024,0.026475,1.705266,1.705472,5.846379,0.005649,-1.494303,0.079533,1.713421,1.715266,5.873931,0.009663,-1.494389
2,N09_M07_F10_K001_11.mat,K001,-0.015085,0.362867,0.363180,8.874512,0.036513,16.424457,0.022838,1.706933,1.707085,5.789899,0.008121,-1.494426,0.082061,1.714590,1.716553,5.537116,0.008039,-1.494616
3,N09_M07_F10_K001_12.mat,K001,-0.015234,0.361718,0.362039,9.432983,0.023483,16.855238,0.024395,1.706094,1.706269,5.818139,0.008204,-1.493631,0.083545,1.714076,1.716110,5.619770,0.008027,-1.494356
4,N09_M07_F10_K001_13.mat,K001,-0.012782,0.361106,0.361332,8.340454,0.164283,15.966562,0.022395,1.707512,1.707659,5.821583,0.008921,-1.494102,0.084894,1.714630,1.716730,5.617704,0.006895,-1.494081


In [3]:

OUTPUT = (
    Path.home()
    / "Desktop"
    / "MachineFailurePredictor"
    / "data"
    / "processed"
)

OUTPUT.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT / "bearing_features.csv"
features_df.to_csv(csv_path, index=False)

print("Saved to:", csv_path)
print("Rows:", len(features_df))
print("Columns:", len(features_df.columns))


Saved to: C:\Users\aryan\Desktop\MachineFailurePredictor\data\processed\bearing_features.csv
Rows: 800
Columns: 20


In [4]:

print("Total feature rows:", len(features_df))
print("Total columns:", len(features_df.columns))

print("\nMissing values:")
print(features_df.isnull().sum())

print("\nBearing IDs:")
print(features_df["bearing_id"].value_counts())

print("\nFirst five rows:")
display(features_df.head())

print("\nSaved CSV exists:", csv_path.exists())
print("CSV path:", csv_path)


Total feature rows: 800
Total columns: 20

Missing values:
file                            0
bearing_id                      0
vibration_1_mean                0
vibration_1_std                 0
vibration_1_rms                 0
vibration_1_peak_to_peak        0
vibration_1_skewness            0
vibration_1_kurtosis            0
phase_current_1_mean            0
phase_current_1_std             0
phase_current_1_rms             0
phase_current_1_peak_to_peak    0
phase_current_1_skewness        0
phase_current_1_kurtosis        0
phase_current_2_mean            0
phase_current_2_std             0
phase_current_2_rms             0
phase_current_2_peak_to_peak    0
phase_current_2_skewness        0
phase_current_2_kurtosis        0
dtype: int64

Bearing IDs:
bearing_id
K001    80
K002    80
K003    80
K004    80
KA01    80
KA03    80
KA04    80
KB23    80
KI01    80
KI03    80
Name: count, dtype: int64

First five rows:


,file,bearing_id,vibration_1_mean,vibration_1_std,vibration_1_rms,vibration_1_peak_to_peak,vibration_1_skewness,vibration_1_kurtosis,phase_current_1_mean,phase_current_1_std,phase_current_1_rms,phase_current_1_peak_to_peak,phase_current_1_skewness,phase_current_1_kurtosis,phase_current_2_mean,phase_current_2_std,phase_current_2_rms,phase_current_2_peak_to_peak,phase_current_2_skewness,phase_current_2_kurtosis
0,N09_M07_F10_K001_1.mat,K001,-0.015454,0.367295,0.367620,9.570312,-0.016718,17.165065,0.023506,1.656556,1.656722,5.730664,0.006718,-1.493523,0.074766,1.666412,1.668088,5.519897,0.008774,-1.494215
1,N09_M07_F10_K001_10.mat,K001,-0.015243,0.362781,0.363101,8.596802,-0.005482,15.989024,0.026475,1.705266,1.705472,5.846379,0.005649,-1.494303,0.079533,1.713421,1.715266,5.873931,0.009663,-1.494389
2,N09_M07_F10_K001_11.mat,K001,-0.015085,0.362867,0.363180,8.874512,0.036513,16.424457,0.022838,1.706933,1.707085,5.789899,0.008121,-1.494426,0.082061,1.714590,1.716553,5.537116,0.008039,-1.494616
3,N09_M07_F10_K001_12.mat,K001,-0.015234,0.361718,0.362039,9.432983,0.023483,16.855238,0.024395,1.706094,1.706269,5.818139,0.008204,-1.493631,0.083545,1.714076,1.716110,5.619770,0.008027,-1.494356
4,N09_M07_F10_K001_13.mat,K001,-0.012782,0.361106,0.361332,8.340454,0.164283,15.966562,0.022395,1.707512,1.707659,5.821583,0.008921,-1.494102,0.084894,1.714630,1.716730,5.617704,0.006895,-1.494081



Saved CSV exists: True
CSV path: C:\Users\aryan\Desktop\MachineFailurePredictor\data\processed\bearing_features.csv


In [5]:

print("Bearing IDs and recording counts:")
display(
    features_df["bearing_id"]
    .value_counts()
    .sort_index()
    .rename_axis("bearing_id")
    .reset_index(name="recordings")
)

print("\nUnique bearing IDs:")
print(sorted(features_df["bearing_id"].unique()))


Bearing IDs and recording counts:


,bearing_id,recordings
0,K001,80
1,K002,80
2,K003,80
3,K004,80
4,KA01,80
5,KA03,80
6,KA04,80
7,KB23,80
8,KI01,80
9,KI03,80



Unique bearing IDs:
['K001', 'K002', 'K003', 'K004', 'KA01', 'KA03', 'KA04', 'KB23', 'KI01', 'KI03']


In [6]:

healthy_ids = {"K001", "K002", "K003", "K004"}

features_df["label"] = features_df["bearing_id"].apply(
    lambda bearing_id: 0 if bearing_id in healthy_ids else 1
)

features_df["condition"] = features_df["label"].map({
    0: "Healthy",
    1: "Damaged"
})

print("Label counts:")
print(features_df["condition"].value_counts())

print("\nLabels by bearing ID:")
display(
    features_df[["bearing_id", "condition"]]
    .drop_duplicates()
    .sort_values("bearing_id")
)


Label counts:
condition
Damaged    480
Healthy    320
Name: count, dtype: int64

Labels by bearing ID:


,bearing_id,condition
0,K001,Healthy
80,K002,Healthy
160,K003,Healthy
240,K004,Healthy
320,KA01,Damaged
400,KA03,Damaged
480,KA04,Damaged
560,KB23,Damaged
640,KI01,Damaged
720,KI03,Damaged


In [7]:

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Use only numerical signal features
feature_columns = [
    col for col in features_df.columns
    if col.startswith((
        "vibration_1_",
        "phase_current_1_",
        "phase_current_2_"
    ))
]

X = features_df[feature_columns].replace(
    [np.inf, -np.inf], np.nan
)

# Remove any rows with invalid feature values
valid = X.notna().all(axis=1)

X = X.loc[valid]
y = features_df.loc[valid, "label"].astype(int)
groups = features_df.loc[valid, "bearing_id"]

print("Feature count:", len(feature_columns))
print("Usable recordings:", len(X))
print("Healthy recordings:", int((y == 0).sum()))
print("Damaged recordings:", int((y == 1).sum()))
print("Distinct bearings:", groups.nunique())

# Each bearing is held out once
logo = LeaveOneGroupOut()
predictions = np.zeros(len(X), dtype=int)

for train_idx, test_idx in logo.split(X, y, groups):
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    )

    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    predictions[test_idx] = model.predict(X.iloc[test_idx])

print("\nLeave-one-bearing-out results")
print("Accuracy:", round(accuracy_score(y, predictions), 4))

print("\nClassification report:")
print(classification_report(
    y,
    predictions,
    labels=[0, 1],
    target_names=["Healthy", "Damaged"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y, predictions, labels=[0, 1]))


Feature count: 18
Usable recordings: 800
Healthy recordings: 320
Damaged recordings: 480
Distinct bearings: 10

Leave-one-bearing-out results
Accuracy: 0.7488

Classification report:
              precision    recall  f1-score   support

     Healthy       0.80      0.50      0.61       320
     Damaged       0.73      0.92      0.81       480

    accuracy                           0.75       800
   macro avg       0.77      0.71      0.71       800
weighted avg       0.76      0.75      0.73       800

Confusion matrix:
[[159 161]
 [ 40 440]]


In [8]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Select the 18 signal features
feature_columns = [
    col for col in features_df.columns
    if col.startswith((
        "vibration_1_",
        "phase_current_1_",
        "phase_current_2_"
    ))
]

X = features_df[feature_columns].replace(
    [np.inf, -np.inf], np.nan
)
valid = X.notna().all(axis=1)

X = X.loc[valid].reset_index(drop=True)
y = features_df.loc[valid, "label"].astype(int).reset_index(drop=True)
groups = features_df.loc[valid, "bearing_id"].reset_index(drop=True)

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=42,
        class_weight="balanced"
    ),
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )
    )
}

logo = LeaveOneGroupOut()
results = []

for name, model in models.items():
    predictions = np.zeros(len(y), dtype=int)

    for train_idx, test_idx in logo.split(X, y, groups):
        model_fold = clone(model)
        model_fold.fit(X.iloc[train_idx], y.iloc[train_idx])
        predictions[test_idx] = model_fold.predict(X.iloc[test_idx])

    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y, predictions), 4),
        "Damaged Precision": round(
            precision_score(y, predictions, pos_label=1, zero_division=0), 4
        ),
        "Damaged Recall": round(
            recall_score(y, predictions, pos_label=1, zero_division=0), 4
        ),
        "Damaged F1": round(
            f1_score(y, predictions, pos_label=1, zero_division=0), 4
        )
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

              Model  Accuracy  Damaged Precision  Damaged Recall  Damaged F1
      Random Forest    0.7488             0.7321          0.9167      0.8141
      Decision Tree    0.6575             0.7309          0.6792      0.7041
Logistic Regression    0.8688             1.0000          0.7812      0.8772


In [9]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    )
)

logo = LeaveOneGroupOut()
probabilities = np.zeros(len(y))

# Predict each bearing using a model that did not train on that bearing
for train_idx, test_idx in logo.split(X, y, groups):
    fold_model = clone(model)
    fold_model.fit(X.iloc[train_idx], y.iloc[train_idx])

    probabilities[test_idx] = fold_model.predict_proba(
        X.iloc[test_idx]
    )[:, 1]

# Compare decision thresholds
rows = []

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    predictions = (probabilities >= threshold).astype(int)

    rows.append({
        "Threshold": threshold,
        "Accuracy": round(accuracy_score(y, predictions), 4),
        "Damaged Precision": round(
            precision_score(y, predictions, zero_division=0), 4
        ),
        "Damaged Recall": round(
            recall_score(y, predictions, zero_division=0), 4
        ),
        "Damaged F1": round(
            f1_score(y, predictions, zero_division=0), 4
        ),
        "Missed Damaged": int(
            ((y == 1) & (predictions == 0)).sum()
        ),
        "False Alarms": int(
            ((y == 0) & (predictions == 1)).sum()
        )
    })

threshold_results = pd.DataFrame(rows)
print(threshold_results.to_string(index=False))

 Threshold  Accuracy  Damaged Precision  Damaged Recall  Damaged F1  Missed Damaged  False Alarms
      0.30    0.8738             0.9847          0.8021      0.8840              95             6
      0.35    0.8762             0.9922          0.8000      0.8858              96             3
      0.40    0.8788             0.9974          0.8000      0.8879              96             1
      0.45    0.8762             1.0000          0.7937      0.8850              99             0
      0.50    0.8688             1.0000          0.7812      0.8772             105             0
      0.55    0.8588             1.0000          0.7646      0.8666             113             0
      0.60    0.8488             1.0000          0.7479      0.8558             121             0
      0.65    0.8462             1.0000          0.7438      0.8530             123             0
      0.70    0.8375             1.0000          0.7292      0.8434             130             0


In [10]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=5000))
])

param_grid = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__class_weight": [None, "balanced"]
}

logo = LeaveOneGroupOut()

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=logo,
    n_jobs=-1
)

search.fit(X, y, groups=groups)

print("Best settings:", search.best_params_)
print("Best cross-validation accuracy:", round(search.best_score_, 4))

# Evaluate the selected settings with the same grouped folds
from sklearn.model_selection import cross_val_predict

best_model = search.best_estimator_

predictions = cross_val_predict(
    best_model,
    X,
    y,
    groups=groups,
    cv=logo,
    n_jobs=-1
)

print("\nGrouped cross-validation accuracy:",
      round(accuracy_score(y, predictions), 4))

print("\nClassification report:")
print(classification_report(
    y,
    predictions,
    labels=[0, 1],
    target_names=["Healthy", "Damaged"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y, predictions, labels=[0, 1]))

Best settings: {'model__C': 0.1, 'model__class_weight': None}
Best cross-validation accuracy: 0.8863

Grouped cross-validation accuracy: 0.8862

Classification report:
              precision    recall  f1-score   support

     Healthy       0.79      0.98      0.87       320
     Damaged       0.98      0.82      0.90       480

    accuracy                           0.89       800
   macro avg       0.89      0.90      0.89       800
weighted avg       0.90      0.89      0.89       800

Confusion matrix:
[[313   7]
 [ 84 396]]


In [11]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf"))
])

param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", 0.01, 0.1, 1],
    "model__class_weight": [None, "balanced"]
}

logo = LeaveOneGroupOut()

search_svm = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="accuracy",
    cv=logo,
    n_jobs=-1
)

search_svm.fit(X, y, groups=groups)

print("Best SVM settings:", search_svm.best_params_)
print("Best grouped CV accuracy:", round(search_svm.best_score_, 4))

from sklearn.model_selection import cross_val_predict

svm_predictions = cross_val_predict(
    search_svm.best_estimator_,
    X,
    y,
    groups=groups,
    cv=logo,
    n_jobs=-1
)

print("\nGrouped accuracy:",
      round(accuracy_score(y, svm_predictions), 4))

print("\nClassification report:")
print(classification_report(
    y,
    svm_predictions,
    labels=[0, 1],
    target_names=["Healthy", "Damaged"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y, svm_predictions, labels=[0, 1]))

Best SVM settings: {'model__C': 1, 'model__class_weight': None, 'model__gamma': 'scale'}
Best grouped CV accuracy: 0.9725

Grouped accuracy: 0.9725

Classification report:
              precision    recall  f1-score   support

     Healthy       0.94      0.99      0.97       320
     Damaged       1.00      0.96      0.98       480

    accuracy                           0.97       800
   macro avg       0.97      0.98      0.97       800
weighted avg       0.97      0.97      0.97       800

Confusion matrix:
[[318   2]
 [ 20 460]]


In [12]:
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf"))
])

param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", 0.01, 0.1, 1],
    "model__class_weight": [None, "balanced"]
}

outer_cv = LeaveOneGroupOut()
predictions = np.zeros(len(y), dtype=int)

for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(X, y, groups), start=1
):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    groups_train = groups.iloc[train_idx]

    inner_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="accuracy",
        cv=LeaveOneGroupOut(),
        n_jobs=-1
    )

    inner_search.fit(
        X_train,
        y_train,
        groups=groups_train
    )

    predictions[test_idx] = inner_search.predict(X.iloc[test_idx])

    print(f"Outer fold {fold}/{groups.nunique()} complete")

print("\nNested grouped CV accuracy:",
      round(accuracy_score(y, predictions), 4))

print("\nClassification report:")
print(classification_report(
    y,
    predictions,
    labels=[0, 1],
    target_names=["Healthy", "Damaged"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y, predictions, labels=[0, 1]))


Outer fold 1/10 complete
Outer fold 2/10 complete
Outer fold 3/10 complete
Outer fold 4/10 complete
Outer fold 5/10 complete
Outer fold 6/10 complete
Outer fold 7/10 complete
Outer fold 8/10 complete
Outer fold 9/10 complete
Outer fold 10/10 complete

Nested grouped CV accuracy: 0.9675

Classification report:
              precision    recall  f1-score   support

     Healthy       0.93      0.99      0.96       320
     Damaged       1.00      0.95      0.97       480

    accuracy                           0.97       800
   macro avg       0.96      0.97      0.97       800
weighted avg       0.97      0.97      0.97       800

Confusion matrix:
[[318   2]
 [ 24 456]]


In [13]:
import numpy as np
import pandas as pd

from scipy.stats import kurtosis, skew
from scipy.signal import welch

def extract_advanced_features(signal, prefix):
    signal = np.asarray(signal, dtype=np.float64).ravel()
    signal = signal[np.isfinite(signal)]

    if len(signal) < 16:
        return {}

    # Remove DC offset for shape and spectral analysis
    centered = signal - np.mean(signal)

    eps = 1e-12
    abs_signal = np.abs(centered)
    rms = np.sqrt(np.mean(centered ** 2))
    mean_abs = np.mean(abs_signal)
    peak = np.max(abs_signal)

    # Time-domain features
    features = {
        f"{prefix}_crest_factor": float(peak / (rms + eps)),
        f"{prefix}_shape_factor": float(rms / (mean_abs + eps)),
        f"{prefix}_impulse_factor": float(peak / (mean_abs + eps)),
        f"{prefix}_peak_amplitude": float(peak),
        f"{prefix}_variance": float(np.var(centered)),
        f"{prefix}_zero_crossing_rate": float(
            np.mean(np.diff(np.signbit(centered)) != 0)
        )
    }

    # Welch spectrum: fs=1 gives normalized frequencies,
    # not frequencies in Hz.
    frequencies, power = welch(
        centered,
        fs=1.0,
        nperseg=min(2048, len(centered))
    )

    power = np.maximum(power, 0)
    total_power = np.sum(power)

    if total_power <= eps:
        features.update({
            f"{prefix}_spectral_centroid": 0.0,
            f"{prefix}_spectral_entropy": 0.0,
            f"{prefix}_spectral_flatness": 0.0,
            f"{prefix}_spectral_high_band_ratio": 0.0
        })
        return features

    probabilities = power / total_power
    nonzero = probabilities > 0

    # Frequency features
    centroid = np.sum(frequencies * probabilities)
    entropy = -np.sum(
        probabilities[nonzero] * np.log(probabilities[nonzero])
    ) / np.log(len(probabilities))

    flatness = np.exp(
        np.mean(np.log(power + eps))
    ) / (np.mean(power) + eps)

    # Relative power in the upper half of the normalized spectrum
    high_band = frequencies >= 0.25
    high_ratio = np.sum(power[high_band]) / total_power

    features.update({
        f"{prefix}_spectral_centroid": float(centroid),
        f"{prefix}_spectral_entropy": float(entropy),
        f"{prefix}_spectral_flatness": float(flatness),
        f"{prefix}_spectral_high_band_ratio": float(high_ratio)
    })

    return features


# Extract advanced features from every recording
advanced_rows = []

for i, path in enumerate(files, start=1):
    try:
        signals = get_signals(path)
        row = {"file": path.name}

        for signal_name in [
            "vibration_1",
            "phase_current_1",
            "phase_current_2"
        ]:
            if signal_name in signals:
                row.update(
                    extract_advanced_features(
                        signals[signal_name],
                        signal_name
                    )
                )

        advanced_rows.append(row)

    except Exception as error:
        print("Skipped:", path.name, "|", repr(error))

    if i % 100 == 0:
        print(f"Processed {i}/{len(files)} recordings")

advanced_df = pd.DataFrame(advanced_rows)

# Keep all existing labels and original features
enhanced_df = features_df.merge(
    advanced_df,
    on="file",
    how="inner",
    validate="one_to_one"
)

print("Enhanced dataset shape:", enhanced_df.shape)
print("Missing values:", int(enhanced_df.isna().sum().sum()))

enhanced_df.to_csv(
    ROOT.parent / "processed" / "bearing_features_enhanced.csv",
    index=False
)

print("Enhanced dataset saved successfully.")

Processed 100/800 recordings
Processed 200/800 recordings
Processed 300/800 recordings
Processed 400/800 recordings
Processed 500/800 recordings
Processed 600/800 recordings
Processed 700/800 recordings
Processed 800/800 recordings
Enhanced dataset shape: (800, 52)
Missing values: 0


OSError: Cannot save file into a non-existent directory: 'C:\Users\aryan\Desktop\MachineFailurePredictor\data\raw\processed'

In [14]:
from pathlib import Path

output_dir = ROOT.parents[1] / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "bearing_features_enhanced.csv"

enhanced_df.to_csv(output_path, index=False)

print("Enhanced dataset saved successfully!")
print("Dataset shape:", enhanced_df.shape)
print("Saved location:", output_path)

Enhanced dataset saved successfully!
Dataset shape: (800, 52)
Saved location: C:\Users\aryan\Desktop\MachineFailurePredictor\data\processed\bearing_features_enhanced.csv


In [15]:
import pandas as pd

check_df = pd.read_csv(
    r"C:\Users\aryan\Desktop\MachineFailurePredictor\data\processed\bearing_features_deduplicated.csv"
)

print("Dataset shape:", check_df.shape)
print("\nRecordings per bearing:")
print(check_df["bearing_id"].value_counts().sort_index())

parts = check_df["file"].str.replace(
    ".mat", "", regex=False
).str.split("_")

check_df["N_condition"] = parts.str[0]
check_df["M_condition"] = parts.str[1]
check_df["F_condition"] = parts.str[2]

for column in ["N_condition", "M_condition", "F_condition"]:
    print(f"\n--- {column} vs machine label ---")
    print(pd.crosstab(
        check_df[column],
        check_df["condition"],
        margins=True
    ))

Dataset shape: (796, 52)

Recordings per bearing:
bearing_id
K001    80
K002    80
K003    80
K004    80
KA01    80
KA03    80
KA04    76
KB23    80
KI01    80
KI03    80
Name: count, dtype: int64

--- N_condition vs machine label ---
condition    Damaged  Healthy  All
N_condition                       
N09              119       80  199
N15              357      240  597
All              476      320  796

--- M_condition vs machine label ---
condition    Damaged  Healthy  All
M_condition                       
M01              120       80  200
M07              356      240  596
All              476      320  796

--- F_condition vs machine label ---
condition    Damaged  Healthy  All
F_condition                       
F04              117       80  197
F10              359      240  599
All              476      320  796


In [16]:
import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Load the cleaned dataset
df = pd.read_csv(
    r"C:\Users\aryan\Desktop\MachineFailurePredictor\data\processed\bearing_features_deduplicated.csv"
)

# Select only model features
excluded = {"file", "bearing_id", "label", "condition"}
feature_columns = [
    col for col in df.columns if col not in excluded
]

X = df[feature_columns].replace(
    [np.inf, -np.inf], np.nan
)
valid_rows = X.notna().all(axis=1)

df = df.loc[valid_rows].reset_index(drop=True)
X = X.loc[valid_rows].reset_index(drop=True)

y = df["label"].astype(int)
groups = df["bearing_id"]

print("Recordings:", len(df))
print("Features:", len(feature_columns))
print("Bearing groups:", groups.nunique())

# Nested grouped cross-validation
outer_cv = LeaveOneGroupOut()
inner_cv = LeaveOneGroupOut()

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf"))
])

param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.01, 0.1, 1],
    "svm__class_weight": [None, "balanced"]
}

predictions = np.zeros(len(df), dtype=int)

for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(X, y, groups), start=1
):
    search = GridSearchCV(
        pipeline,
        param_grid,
        cv=inner_cv,
        scoring="accuracy",
        n_jobs=-1
    )

    search.fit(
        X.iloc[train_idx],
        y.iloc[train_idx],
        groups=groups.iloc[train_idx]
    )

    predictions[test_idx] = search.predict(X.iloc[test_idx])

    print(
        f"Fold {fold}/{groups.nunique()} completed | "
        f"Held-out bearing: {groups.iloc[test_idx].iloc[0]}"
    )

print("\nNested grouped CV accuracy:",
      round(accuracy_score(y, predictions), 4))

print("\nClassification report:")
print(classification_report(
    y, predictions,
    labels=[0, 1],
    target_names=["Healthy", "Damaged"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y, predictions, labels=[0, 1]))

Recordings: 796
Features: 48
Bearing groups: 10
Fold 1/10 completed | Held-out bearing: K001
Fold 2/10 completed | Held-out bearing: K002
Fold 3/10 completed | Held-out bearing: K003
Fold 4/10 completed | Held-out bearing: K004
Fold 5/10 completed | Held-out bearing: KA01
Fold 6/10 completed | Held-out bearing: KA03
Fold 7/10 completed | Held-out bearing: KA04
Fold 8/10 completed | Held-out bearing: KB23
Fold 9/10 completed | Held-out bearing: KI01
Fold 10/10 completed | Held-out bearing: KI03

Nested grouped CV accuracy: 0.9975

Classification report:
              precision    recall  f1-score   support

     Healthy       1.00      0.99      1.00       320
     Damaged       1.00      1.00      1.00       476

    accuracy                           1.00       796
   macro avg       1.00      1.00      1.00       796
weighted avg       1.00      1.00      1.00       796

Confusion matrix:
[[318   2]
 [  0 476]]


In [17]:
# Compare the original 18 features against the enhanced 48 features

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

# Load the cleaned dataset
df = pd.read_csv(
    r"C:\Users\aryan\Desktop\MachineFailurePredictor\data\processed\bearing_features_deduplicated.csv"
)

# Get the original feature names from features_df
excluded = {"file", "bearing_id", "label", "condition"}
original_features = [
    col for col in features_df.columns
    if col not in excluded
]

print("Original feature count:", len(original_features))

if len(original_features) != 18:
    raise ValueError(
        "Expected 18 original features. Check features_df columns."
    )

X = df[original_features].replace(
    [np.inf, -np.inf], np.nan
)
valid = X.notna().all(axis=1)

df = df.loc[valid].reset_index(drop=True)
X = X.loc[valid].reset_index(drop=True)
y = df["label"].astype(int)
groups = df["bearing_id"]

outer_cv = LeaveOneGroupOut()
inner_cv = LeaveOneGroupOut()

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf"))
])

param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.01, 0.1, 1],
    "svm__class_weight": [None, "balanced"]
}

predictions = np.zeros(len(df), dtype=int)

for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(X, y, groups), start=1
):
    search = GridSearchCV(
        pipeline,
        param_grid,
        cv=inner_cv,
        scoring="accuracy",
        n_jobs=-1
    )

    search.fit(
        X.iloc[train_idx],
        y.iloc[train_idx],
        groups=groups.iloc[train_idx]
    )

    predictions[test_idx] = search.predict(X.iloc[test_idx])
    print(f"Fold {fold}/10 completed")

print("\nOriginal 18-feature model")
print("Accuracy:", round(accuracy_score(y, predictions), 4))
print("Confusion matrix:")
print(confusion_matrix(y, predictions, labels=[0, 1]))

Original feature count: 18
Fold 1/10 completed
Fold 2/10 completed
Fold 3/10 completed
Fold 4/10 completed
Fold 5/10 completed
Fold 6/10 completed
Fold 7/10 completed
Fold 8/10 completed
Fold 9/10 completed
Fold 10/10 completed

Original 18-feature model
Accuracy: 0.9673
Confusion matrix:
[[318   2]
 [ 24 452]]


In [18]:
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.model_selection import GridSearchCV, LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# 1. Load the cleaned dataset
data_path = Path(
    r"C:\Users\aryan\Desktop\MachineFailurePredictor"
    r"\data\processed\bearing_features_deduplicated.csv"
)

df = pd.read_csv(data_path)

# 2. Select the 48 features
excluded = {"file", "bearing_id", "label", "condition"}
feature_columns = [
    col for col in df.columns if col not in excluded
]

X = df[feature_columns].replace([np.inf, -np.inf], np.nan)
valid = X.notna().all(axis=1)

df = df.loc[valid].reset_index(drop=True)
X = X.loc[valid].reset_index(drop=True)

y = df["label"].astype(int)
groups = df["bearing_id"]

print("Training recordings:", len(X))
print("Number of features:", len(feature_columns))

# 3. Define the model pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf"))
])

param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.01, 0.1, 1],
    "svm__class_weight": [None, "balanced"]
}

# 4. Select hyperparameters using grouped cross-validation
cv = LeaveOneGroupOut()

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    refit=True
)

search.fit(X, y, groups=groups)

# 5. Save the fitted model and its feature list
project_dir = Path(
    r"C:\Users\aryan\Desktop\MachineFailurePredictor"
)
models_dir = project_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

model_path = models_dir / "machine_failure_svm.joblib"
features_path = models_dir / "feature_columns.json"
metadata_path = models_dir / "model_metadata.json"

joblib.dump(search.best_estimator_, model_path)

with open(features_path, "w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=2)

metadata = {
    "model": "SVM with RBF kernel",
    "feature_count": len(feature_columns),
    "training_recordings": len(X),
    "bearing_groups": int(groups.nunique()),
    "best_parameters": search.best_params_,
    "classes": {"0": "Healthy", "1": "Damaged"},
    "dataset": "Paderborn bearing dataset, deduplicated"
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("\nFinal candidate model trained and saved!")
print("Best parameters:", search.best_params_)
print("Model:", model_path)
print("Feature list:", features_path)
print("Metadata:", metadata_path)

Training recordings: 796
Number of features: 48

Final candidate model trained and saved!
Best parameters: {'svm__C': 1, 'svm__class_weight': 'balanced', 'svm__gamma': 'scale'}
Model: C:\Users\aryan\Desktop\MachineFailurePredictor\models\machine_failure_svm.joblib
Feature list: C:\Users\aryan\Desktop\MachineFailurePredictor\models\feature_columns.json
Metadata: C:\Users\aryan\Desktop\MachineFailurePredictor\models\model_metadata.json


In [19]:
import joblib
import json
import pandas as pd
from pathlib import Path

# Project paths
PROJECT_DIR = Path(
    r"C:\Users\aryan\Desktop\MachineFailurePredictor"
)

# Load saved model and feature list
model = joblib.load(
    PROJECT_DIR / "models" / "machine_failure_svm.joblib"
)

with open(
    PROJECT_DIR / "models" / "feature_columns.json",
    "r",
    encoding="utf-8"
) as f:
    feature_columns = json.load(f)

# Load cleaned dataset
df = pd.read_csv(
    PROJECT_DIR / "data" / "processed"
    / "bearing_features_deduplicated.csv"
)

# Select the exact features in the required order
X = df[feature_columns]

# Predict the first 10 recordings
predictions = model.predict(X.iloc[:10])

# Display actual and predicted results
results = pd.DataFrame({
    "Bearing_ID": df["bearing_id"].iloc[:10].values,
    "Actual": df["label"].iloc[:10].map({
        0: "Healthy",
        1: "Damaged"
    }).values,
    "Predicted": pd.Series(predictions).map({
        0: "Healthy",
        1: "Damaged"
    }).values
})

display(results)

,Bearing_ID,Actual,Predicted
0,K001,Healthy,Healthy
1,K001,Healthy,Healthy
2,K001,Healthy,Healthy
3,K001,Healthy,Healthy
4,K001,Healthy,Healthy
5,K001,Healthy,Healthy
6,K001,Healthy,Healthy
7,K001,Healthy,Healthy
8,K001,Healthy,Healthy
9,K001,Healthy,Healthy


In [20]:
# Select 5 healthy and 5 damaged recordings
healthy_samples = df[df["label"] == 0].head(5)
damaged_samples = df[df["label"] == 1].head(5)

test_samples = pd.concat(
    [healthy_samples, damaged_samples]
).reset_index(drop=True)

# Predict using the saved model
X_test = test_samples[feature_columns]
predictions = model.predict(X_test)

# Display results
results = pd.DataFrame({
    "Bearing_ID": test_samples["bearing_id"].values,
    "Actual": test_samples["label"].map({
        0: "Healthy",
        1: "Damaged"
    }).values,
    "Predicted": pd.Series(predictions).map({
        0: "Healthy",
        1: "Damaged"
    }).values
})

results["Correct"] = (
    results["Actual"] == results["Predicted"]
)

display(results)

,Bearing_ID,Actual,Predicted,Correct
0,K001,Healthy,Healthy,True
1,K001,Healthy,Healthy,True
2,K001,Healthy,Healthy,True
3,K001,Healthy,Healthy,True
4,K001,Healthy,Healthy,True
5,KA01,Damaged,Damaged,True
6,KA01,Damaged,Damaged,True
7,KA01,Damaged,Damaged,True
8,KA01,Damaged,Damaged,True
9,KA01,Damaged,Damaged,True


In [22]:
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import (
    LeaveOneGroupOut,
    GridSearchCV
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Load cleaned dataset
df = pd.read_csv(
    r"C:\Users\aryan\Desktop\MachineFailurePredictor"
    r"\data\processed\bearing_features_deduplicated.csv"
)

# Select the 48 numeric features in their original order
excluded = {"file", "bearing_id", "label", "condition"}
feature_columns = [
    col for col in df.columns if col not in excluded
]

X = df[feature_columns].replace([np.inf, -np.inf], np.nan)
valid = X.notna().all(axis=1)

df = df.loc[valid].reset_index(drop=True)
X = X.loc[valid].reset_index(drop=True)
y = df["label"].astype(int)
groups = df["bearing_id"]

# Model pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf"))
])

# Same parameter grid as the earlier evaluation
param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.01, 0.1, 1],
    "svm__class_weight": [None, "balanced"]
}

outer_cv = LeaveOneGroupOut()

# Store the prediction for every held-out recording
y_true_cv = np.zeros(len(y), dtype=int)
y_pred_cv = np.zeros(len(y), dtype=int)

for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(X, y, groups), start=1
):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    train_groups = groups.iloc[train_idx]

    inner_cv = LeaveOneGroupOut()

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=inner_cv,
        scoring="accuracy",
        n_jobs=-1,
        refit=True
    )

    search.fit(X_train, y_train, groups=train_groups)

    y_true_cv[test_idx] = y_test.to_numpy()
    y_pred_cv[test_idx] = search.predict(X_test)

    print(f"Completed fold {fold}/10")

# Final evaluation
print("\nNested Grouped Cross-Validation Results")
print("Accuracy:", round(
    accuracy_score(y_true_cv, y_pred_cv), 4
))

print("\nClassification Report:")
print(classification_report(
    y_true_cv,
    y_pred_cv,
    labels=[0, 1],
    target_names=["Healthy", "Damaged"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(
    y_true_cv,
    y_pred_cv,
    labels=[0, 1]
))

Completed fold 1/10
Completed fold 2/10
Completed fold 3/10
Completed fold 4/10
Completed fold 5/10
Completed fold 6/10
Completed fold 7/10
Completed fold 8/10
Completed fold 9/10
Completed fold 10/10

Nested Grouped Cross-Validation Results
Accuracy: 0.9975

Classification Report:
              precision    recall  f1-score   support

     Healthy       1.00      0.99      1.00       320
     Damaged       1.00      1.00      1.00       476

    accuracy                           1.00       796
   macro avg       1.00      1.00      1.00       796
weighted avg       1.00      1.00      1.00       796

Confusion Matrix:
[[318   2]
 [  0 476]]


In [23]:
import json
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

project_dir = Path(
    r"C:\Users\aryan\Desktop\MachineFailurePredictor"
)

results_dir = project_dir / "reports"
results_dir.mkdir(parents=True, exist_ok=True)

# Save each recording's cross-validation prediction
predictions_df = pd.DataFrame({
    "bearing_id": df["bearing_id"].values,
    "actual_label": y_true_cv,
    "predicted_label": y_pred_cv
})

predictions_df["actual_condition"] = (
    predictions_df["actual_label"]
    .map({0: "Healthy", 1: "Damaged"})
)

predictions_df["predicted_condition"] = (
    predictions_df["predicted_label"]
    .map({0: "Healthy", 1: "Damaged"})
)

predictions_df["correct"] = (
    predictions_df["actual_label"]
    == predictions_df["predicted_label"]
)

predictions_df.to_csv(
    results_dir / "nested_cv_predictions.csv",
    index=False
)

# Save evaluation metrics
report = classification_report(
    y_true_cv,
    y_pred_cv,
    labels=[0, 1],
    target_names=["Healthy", "Damaged"],
    output_dict=True,
    zero_division=0
)

metrics = {
    "evaluation": "Nested Leave-One-Group-Out CV",
    "accuracy": float(accuracy_score(y_true_cv, y_pred_cv)),
    "recordings": len(y_true_cv),
    "features": 48,
    "bearing_groups": int(df["bearing_id"].nunique()),
    "confusion_matrix": confusion_matrix(
        y_true_cv, y_pred_cv, labels=[0, 1]
    ).tolist(),
    "classification_report": report
}

with open(
    results_dir / "evaluation_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metrics, f, indent=2)

print("Evaluation files saved successfully!")
print("Predictions:", results_dir / "nested_cv_predictions.csv")
print("Metrics:", results_dir / "evaluation_metrics.json")

Evaluation files saved successfully!
Predictions: C:\Users\aryan\Desktop\MachineFailurePredictor\reports\nested_cv_predictions.csv
Metrics: C:\Users\aryan\Desktop\MachineFailurePredictor\reports\evaluation_metrics.json


In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Load the deduplicated dataset
df = pd.read_csv(
    r"C:\Users\aryan\Desktop\MachineFailurePredictor\data\processed\bearing_features_deduplicated.csv"
)

# Select only model features; exclude metadata and labels
excluded = {"file", "bearing_id", "label", "condition"}
feature_columns = [c for c in df.columns if c not in excluded]

X = df[feature_columns].replace([np.inf, -np.inf], np.nan)
valid_rows = X.notna().all(axis=1)

df = df.loc[valid_rows].reset_index(drop=True)
X = X.loc[valid_rows].reset_index(drop=True)

y = df["label"].astype(int)
groups = df["bearing_id"]

print("Recordings:", len(df))
print("Features:", len(feature_columns))
print("Bearing groups:", groups.nunique())
print("\nClass distribution:")
print(df["condition"].value_counts())

# Outer CV: leave one entire bearing out
outer_cv = LeaveOneGroupOut()

# Inner CV: tune using only training bearings
inner_cv = LeaveOneGroupOut()

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf"))
])

param_grid = {
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.01, 0.1, 1],
    "svm__class_weight": [None, "balanced"]
}

predictions = np.zeros(len(df), dtype=int)
fold_results = []

for fold, (train_idx, test_idx) in enumerate(
    outer_cv.split(X, y, groups), start=1
):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train = groups.iloc[train_idx]

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=inner_cv,
        scoring="accuracy",
        n_jobs=-1
    )

    search.fit(X_train, y_train, groups=groups_train)
    pred = search.predict(X_test)
    predictions[test_idx] = pred

    fold_results.append({
        "held_out_bearing": groups.iloc[test_idx].iloc[0],
        "test_recordings": len(test_idx),
        "accuracy": accuracy_score(y_test, pred),
        "best_params": search.best_params_
    })

print("\nNested grouped CV accuracy:",
      round(accuracy_score(y, predictions), 4))

print("\nClassification report:")
print(classification_report(
    y, predictions,
    labels=[0, 1],
    target_names=["Healthy", "Damaged"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y, predictions, labels=[0, 1]))

print("\nAccuracy by held-out bearing:")
print(pd.DataFrame(fold_results)[
    ["held_out_bearing", "test_recordings", "accuracy"]
].to_string(index=False))
